# 04 · Limpieza Avanzada — MATRICULADOS (procesamiento por fases)

**Objetivo:** aplicar a `matriculados_clean.parquet` todas las correcciones planificadas y generar `matriculados_clean_v2.parquet`, dejando la capa Silver **inmaculada** antes de pasar a Gold.

**Entrada:** `data/Silver/matriculados_clean.parquet` (~17.37 M filas, ~736 MB)
**Salida:** `data/Silver/matriculados_clean_v2.parquet`

**Dataset:** semestral (granularidad real persona × programa × local × semestre). Clave de **producción** (7 cols): `PERIODO_ESTANDARIZADO + CODIGO_INEI + CODIGO_SIU_PROGRAMA + CODIGO_GRUPO_1 + CODIGO_GRUPO_3 + CODIGO_LOCAL + GUID_PERSONA`. Sobre el archivo original esta clave da **0 duplicados** (validado en `02_experimento_matriculados_duplicados.ipynb`).

---

## Estrategia de procesamiento por fases (para bajar el pico de memoria)

El pipeline único (join + imputaciones + normalización de 26 columnas + regex) excedía los 10 GB de pico en `sink_parquet`. Se divide en **dos fases** para que cada una materialice solo un subconjunto de transformaciones:

### FASE A · Limpieza estructural (sin normalización ni discapacidad)
Pipeline lazy con: join con `df_mapeo`, imputación de grupos (mapa + centinela), imputación de textos y de `ANIO_PERIODO_INGRESO`. **No** incluye normalización de textos ni transformación de discapacidad. Guarda en un archivo **intermedio**: `data/Silver/temp_matriculados_stage1.parquet`.

### FASE B · Normalización y discapacidad (sobre el archivo intermedio)
Carga el archivo intermedio, aplica **solo** la normalización de textos y la transformación de discapacidad, y guarda el resultado final. Al terminar, **elimina el archivo intermedio**.

### Optimizaciones de memoria
- **Chunk de streaming:** 32 MB (en lugar de 64 MB) para lotes más pequeños.
- **Hilos:** `POLARS_MAX_THREADS=12` (antes de importar Polars).
- **Normalización selectiva:** la regex `\s+ → ' '` solo se aplica a las columnas que **realmente** tienen espacios dobles (según diagnóstico: `NOMBRE_PROGRAMA` y posiblemente `NOMBRE_ENTIDAD`); el resto solo recibe `strip_chars()` (operación ligera, sin regex).

> **Pico de memoria objetivo:** < 6 GB (vs > 10 GB del pipeline único).
> ⚠️ Este notebook es **autónomo**: procesa únicamente Matriculados y **no depende de Ingresantes**.

In [1]:
# Límite de hilos ANTES de importar Polars (12 núcleos)
import os
os.environ["POLARS_MAX_THREADS"] = "12"

import gc
from pathlib import Path

import polars as pl

pl.Config.set_streaming_chunk_size(32 * 1024 * 1024)  # 32 MB por lote de streaming

print("polars", pl.__version__)
print("hilos activos:", pl.thread_pool_size())


def rss_actual_gb():
    """RSS actual del proceso en GB (Linux, /proc/self/statm)."""
    try:
        with open("/proc/self/statm", encoding="utf-8") as fh:
            paginas = int(fh.read().split()[1])
        return paginas * os.sysconf("SC_PAGE_SIZE") / (1024**3)
    except (OSError, ValueError, IndexError):
        return float("nan")


print(f"RSS inicial: {rss_actual_gb():.2f} GB")

polars 1.44.1
hilos activos: 12
RSS inicial: 0.08 GB


In [2]:
# Rutas del proyecto (misma detección automática que los notebooks 01–03)
current_dir = Path.cwd()
if (current_dir / "data").exists():
    PROJECT_ROOT = current_dir
elif (current_dir.parent / "data").exists():
    PROJECT_ROOT = current_dir.parent
else:
    raise FileNotFoundError(
        "No se encontró la carpeta 'data'. Ejecuta este notebook desde la raíz "
        "del proyecto o desde notebooks/."
    )

SILVER = PROJECT_ROOT / "data" / "Silver"
MAT_PATH = SILVER / "matriculados_clean.parquet"
MAT_STAGE1 = SILVER / "temp_matriculados_stage1.parquet"
MAT_V2 = SILVER / "matriculados_clean_v2.parquet"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Entrada existe:", MAT_PATH.exists(), "→", MAT_PATH)

PROJECT_ROOT: /mnt/datos/Proyectos/A.Prueba Tecnica UCSP
Entrada existe: True → /mnt/datos/Proyectos/A.Prueba Tecnica UCSP/data/Silver/matriculados_clean.parquet


## FASE 0 · Esquema y verificación de duplicados (archivo original)

Se lee el esquema con `collect_schema()` (metadatos, sin cargar datos) para identificar columnas **Utf8** (normalización) y **`DES_DISCAPACIDAD_*`** (compactación).

Se verifica que la **clave de producción (K7)** da **0 duplicados** sobre el archivo original mediante una agregación en streaming (sin materializar el dataset).

In [3]:
# Clave de producción (7 columnas) y columnas relevantes
GRUPO_COLS = ["CODIGO_GRUPO_1", "NOMBRE_GRUPO_1", "CODIGO_GRUPO_3", "NOMBRE_GRUPO_3"]
CLAVE_MAT_PROD = [
    "PERIODO_ESTANDARIZADO",
    "CODIGO_INEI",
    "CODIGO_SIU_PROGRAMA",
    "CODIGO_GRUPO_1",
    "CODIGO_GRUPO_3",
    "CODIGO_LOCAL",
    "GUID_PERSONA",
]

# Esquema del archivo original (solo metadatos)
schema_orig = pl.scan_parquet(MAT_PATH).collect_schema()
STRING_COLS = [c for c, t in schema_orig.items() if t == pl.Utf8]
DISC_COLS = [c for c in schema_orig.names() if c.startswith("DES_DISCAPACIDAD")]

print(f"Columnas totales: {len(schema_orig.names())}")
print(f"Columnas Utf8 a normalizar ({len(STRING_COLS)}):", STRING_COLS)
print(f"Columnas de discapacidad ({len(DISC_COLS)}):", DISC_COLS)

Columnas totales: 40
Columnas Utf8 a normalizar (26): ['CODIGO_INEI', 'NOMBRE_ENTIDAD', 'TIPO_ENTIDAD', 'TIPO_GESTION', 'TIPO_CONSTITUCION', 'LICENCIA', 'PERIODO', 'PERIODO_ESTANDARIZADO', 'NIVEL_ACADEMICO', 'PERIODO_LECTIVO', 'NOMBRE_GRUPO_1', 'NOMBRE_GRUPO_3', 'NOMBRE_PROGRAMA', 'ES_LOCAL_PRINCIPAL', 'CODIGO_LOCAL', 'DEPARTAMENTO_LOCAL', 'PROVINCIA_LOCAL', 'DISTRITO_LOCAL', 'GUID_PERSONA', 'SEXO', 'EDAD', 'NACIONALIDAD', 'DEPARTAMENTO_NACIMIENTO', 'FECHA_INICIO_PERIODO', 'FECHA_FIN_PERIODO', 'CERT_GRAVEDAD']
Columnas de discapacidad (7): ['DES_DISCAPACIDAD_DE_COMUNICACION', 'DES_DISCAPACIDAD_DE_CONDUCTA', 'DES_DISCAPACIDAD_DE_DESTREZA', 'DES_DISCAPACIDAD_DE_DISPOSICION', 'DES_DISCAPACIDAD_DE_LOCOMOCION', 'DES_DISCAPACIDAD_DE_SITUACION', 'DES_DISCAPACIDAD_DEL_CUIDADO']


In [4]:
# Verificación de duplicados sobre el ARCHIVO ORIGINAL (clave de producción K7)
lf_orig = pl.scan_parquet(MAT_PATH)
n_orig = lf_orig.select(pl.len()).collect().item()

dup_k7_orig = (
    lf_orig
    .group_by(CLAVE_MAT_PROD)
    .len()
    .filter(pl.col("len") > 1)
    .select(pl.col("len").sum())
    .collect()
    .item()
)

print(f"Filas totales (original): {n_orig:,}")
print(f"Duplicados bajo clave de producción K7 (original): {dup_k7_orig:,}")
assert dup_k7_orig == 0, "Se esperaban 0 duplicados bajo la clave de producción"
print("✅ MATRICULADOS (original): 0 duplicados bajo la clave de producción. No se elimina nada.")

Filas totales (original): 17,368,424
Duplicados bajo clave de producción K7 (original): 0
✅ MATRICULADOS (original): 0 duplicados bajo la clave de producción. No se elimina nada.


---
## FASE A · Limpieza estructural

### A.1 · Mapeo de grupos (diccionario de la verdad)

Consulta **ligera e independiente**: se cargan solo las columnas necesarias, se filtran las filas **sin nulos** en `CODIGO_GRUPO_1` y `CODIGO_GRUPO_3`, y se conserva un registro por `CODIGO_SIU_PROGRAMA`. Resultado: DataFrame **minúsculo** (~520 filas) usado como tabla de referencia en el join. Las columnas se renombran con sufijo `_MAP` para no colisionar con las originales.

In [5]:
mapeo_cols = ["CODIGO_SIU_PROGRAMA"] + GRUPO_COLS

df_mapeo = (
    pl.scan_parquet(MAT_PATH)
    .select(mapeo_cols)
    .filter(pl.col("CODIGO_GRUPO_1").is_not_null() & pl.col("CODIGO_GRUPO_3").is_not_null())
    .unique(subset=["CODIGO_SIU_PROGRAMA"])
    .collect()
    .rename({c: f"{c}_MAP" for c in GRUPO_COLS})
)

print(f"Mapeo construido para {df_mapeo.height:,} programas")
print(df_mapeo.head())

Mapeo construido para 520 programas
shape: (5, 5)
┌───────────────────┬───────────────────┬───────────────────┬───────────────────┬──────────────────┐
│ CODIGO_SIU_PROGRA ┆ CODIGO_GRUPO_1_MA ┆ NOMBRE_GRUPO_1_MA ┆ CODIGO_GRUPO_3_MA ┆ NOMBRE_GRUPO_3_M │
│ MA                ┆ P                 ┆ P                 ┆ P                 ┆ AP               │
│ ---               ┆ ---               ┆ ---               ┆ ---               ┆ ---              │
│ i64               ┆ f64               ┆ str               ┆ f64               ┆ str              │
╞═══════════════════╪═══════════════════╪═══════════════════╪═══════════════════╪══════════════════╡
│ 396               ┆ 9.0               ┆ SALUD Y BIENESTAR ┆ 918.0             ┆ NUTRICION        │
│ 280               ┆ 5.0               ┆ CIENCIAS          ┆ 511.0             ┆ BIOLOGIA         │
│                   ┆                   ┆ NATURALES,        ┆                   ┆                  │
│                   ┆                   ┆

### A.2 · Pipeline lazy estructural → `temp_matriculados_stage1.parquet`

Solo las transformaciones **estructurales** (baratas en memoria, sin regex):
1. **Join left** con `df_mapeo`.
2. **Nivel 1 (mapa):** rellenar nulos de grupos con los valores `_MAP`; eliminar columnas `_MAP`.
3. **Nivel 2 (centinela):** residuales → `-1` / `"SIN CLASIFICACION"`.
4. **Textos:** `DEPARTAMENTO_NACIMIENTO` y `NACIONALIDAD` → `"NO ESPECIFICADO"`.
5. **Año de ingreso:** `ANIO_PERIODO_INGRESO` ← año de `PERIODO_ESTANDARIZADO`.

Se guarda con `sink_parquet` (streaming) en el archivo intermedio. **No** se normalizan textos ni se toca la discapacidad aquí.

In [6]:
lf_a = (
    pl.scan_parquet(MAT_PATH)
    # 1. Join con el diccionario de la verdad
    .join(df_mapeo.lazy(), on="CODIGO_SIU_PROGRAMA", how="left")
    # 2. Nivel 1 · Rellenar nulos con los valores del mapa
    .with_columns([pl.col(f"{c}_MAP").fill_null(pl.col(c)).alias(c) for c in GRUPO_COLS])
    .drop([f"{c}_MAP" for c in GRUPO_COLS])
    # 3. Nivel 2 · Centinela para los residuales
    .with_columns([
        pl.col("CODIGO_GRUPO_1").fill_null(-1).alias("CODIGO_GRUPO_1"),
        pl.col("CODIGO_GRUPO_3").fill_null(-1).alias("CODIGO_GRUPO_3"),
        pl.col("NOMBRE_GRUPO_1").fill_null("SIN CLASIFICACION").alias("NOMBRE_GRUPO_1"),
        pl.col("NOMBRE_GRUPO_3").fill_null("SIN CLASIFICACION").alias("NOMBRE_GRUPO_3"),
    ])
    # 4. Imputación de textos
    .with_columns([
        pl.col("DEPARTAMENTO_NACIMIENTO").fill_null("NO ESPECIFICADO").alias("DEPARTAMENTO_NACIMIENTO"),
        pl.col("NACIONALIDAD").fill_null("NO ESPECIFICADO").alias("NACIONALIDAD"),
    ])
    # 5. Imputación del año de ingreso desde el periodo
    .with_columns(
        pl.col("ANIO_PERIODO_INGRESO")
        .fill_null(pl.col("PERIODO_ESTANDARIZADO").str.slice(0, 4).cast(pl.Int64))
        .alias("ANIO_PERIODO_INGRESO")
    )
)

MAT_STAGE1.unlink(missing_ok=True)
lf_a.sink_parquet(MAT_STAGE1)
print(f"Guardado intermedio: {MAT_STAGE1}")
print(f"Tamaño en disco: {MAT_STAGE1.stat().st_size / 1e9:.2f} GB")
print(f"RSS tras Fase A: {rss_actual_gb():.2f} GB")

Guardado intermedio: /mnt/datos/Proyectos/A.Prueba Tecnica UCSP/data/Silver/temp_matriculados_stage1.parquet
Tamaño en disco: 0.44 GB
RSS tras Fase A: 0.41 GB


In [7]:
# Liberar referencias pesadas de la Fase A (el archivo intermedio ya está en disco)
del lf_a, df_mapeo
gc.collect()
print(f"RSS tras liberar Fase A: {rss_actual_gb():.2f} GB")

RSS tras liberar Fase A: 0.27 GB


---
## FASE B · Normalización y discapacidad

### B.1 · Detectar columnas con espacios dobles

Se escanea el archivo intermedio y se cuenta, por columna Utf8, cuántas filas contienen **dos o más espacios seguidos** (`\s{2,}`). Solo esas columnas reciben la regex `\s+ → ' '`; el resto solo `strip_chars()` (ligero, sin regex). Según el diagnóstico, son `NOMBRE_PROGRAMA` y posiblemente `NOMBRE_ENTIDAD`.

In [8]:
lf_stage = pl.scan_parquet(MAT_STAGE1)

# Contar filas con 2+ espacios seguidos por columna Utf8 (agregación streaming, resultado pequeño)
conteos_dobles = (
    lf_stage
    .select([pl.col(c).str.contains(r"\s{2,}").sum().alias(c) for c in STRING_COLS])
    .collect()
    .row(0)
)

COLS_CON_DOBLES = [c for c, n in zip(STRING_COLS, conteos_dobles) if n > 0]
COLS_SIN_DOBLES = [c for c in STRING_COLS if c not in COLS_CON_DOBLES]

print(f"Columnas con espacios dobles ({len(COLS_CON_DOBLES)}):", COLS_CON_DOBLES)
print(f"Columnas solo con strip ({len(COLS_SIN_DOBLES)}):", COLS_SIN_DOBLES)
print("Conteos de filas con 2+ espacios:", {c: int(n) for c, n in zip(STRING_COLS, conteos_dobles) if n > 0})

Columnas con espacios dobles (1): ['NOMBRE_PROGRAMA']
Columnas solo con strip (25): ['CODIGO_INEI', 'NOMBRE_ENTIDAD', 'TIPO_ENTIDAD', 'TIPO_GESTION', 'TIPO_CONSTITUCION', 'LICENCIA', 'PERIODO', 'PERIODO_ESTANDARIZADO', 'NIVEL_ACADEMICO', 'PERIODO_LECTIVO', 'NOMBRE_GRUPO_1', 'NOMBRE_GRUPO_3', 'ES_LOCAL_PRINCIPAL', 'CODIGO_LOCAL', 'DEPARTAMENTO_LOCAL', 'PROVINCIA_LOCAL', 'DISTRITO_LOCAL', 'GUID_PERSONA', 'SEXO', 'EDAD', 'NACIONALIDAD', 'DEPARTAMENTO_NACIMIENTO', 'FECHA_INICIO_PERIODO', 'FECHA_FIN_PERIODO', 'CERT_GRAVEDAD']
Conteos de filas con 2+ espacios: {'NOMBRE_PROGRAMA': 56}


### B.2 · Normalización selectiva y transformación de discapacidad

Sobre el archivo intermedio:
1. **Normalización:**
   - Columnas **con espacios dobles**: `strip_chars()` + `replace_all(r'\s+', ' ')` (regex necesaria para colapsar).
   - Columnas **sin espacios dobles**: solo `strip_chars()` (operación ligera, sin regex).
2. **Discapacidad:** `TIENE_DISCAPACIDAD` (bool) y eliminación de las 7 columnas `DES_DISCAPACIDAD_*`.

Se guarda el resultado final con `sink_parquet`. Si por compatibilidad `sink_parquet` fallara, se usa `collect(engine="streaming").write_parquet` como respaldo (misma semántica).

In [9]:
# Expresiones de normalización: regex solo donde hay espacios dobles, strip en el resto
norm_exprs = [
    pl.col(c).str.strip_chars().str.replace_all(r"\s+", " ").alias(c) if c in COLS_CON_DOBLES
    else pl.col(c).str.strip_chars().alias(c)
    for c in STRING_COLS
]

lf_b = (
    pl.scan_parquet(MAT_STAGE1)
    # 1. Normalización selectiva de textos
    .with_columns(norm_exprs)
    # 2. Compactar discapacidad y eliminar columnas originales
    .with_columns(
        pl.any_horizontal(pl.col("^DES_DISCAPACIDAD_.*$").is_not_null()).alias("TIENE_DISCAPACIDAD")
    )
    .drop(DISC_COLS)
)

print("Pipeline B definido.")
print("Esquema resultante (número de columnas):", len(lf_b.collect_schema().names()))

Pipeline B definido.
Esquema resultante (número de columnas): 34


In [10]:
# Guardado final (streaming). Respaldo con collect+write si sink_parquet no fuera estable
MAT_V2.unlink(missing_ok=True)
try:
    lf_b.sink_parquet(MAT_V2)
    print("Guardado con sink_parquet (streaming).")
except Exception as e:
    print(f"sink_parquet falló ({type(e).__name__}); usando collect(engine='streaming').write_parquet")
    df_b = lf_b.collect(engine="streaming")
    df_b.write_parquet(MAT_V2)
    del df_b
    gc.collect()

print(f"Guardado final: {MAT_V2}")
print(f"Tamaño en disco: {MAT_V2.stat().st_size / 1e9:.2f} GB")
print(f"RSS tras Fase B: {rss_actual_gb():.2f} GB")

Guardado con sink_parquet (streaming).
Guardado final: /mnt/datos/Proyectos/A.Prueba Tecnica UCSP/data/Silver/matriculados_clean_v2.parquet
Tamaño en disco: 0.44 GB
RSS tras Fase B: 0.61 GB


---
## FASE 4 · Validaciones finales

Se valida el archivo **V2** con lecturas ligeras (`scan_parquet` + agregaciones en streaming):
1. **Duplicados K7:** original = 0; en V2 se reporta el conteo real (puede ser > 0 por colisiones de nulos tras la imputación de grupos).
2. **Nulos en columnas imputadas:** grupos, textos y `ANIO_PERIODO_INGRESO` deben quedar en 0.
3. **Espacios:** 0 en bordes y 0 dobles en las columnas Utf8.

In [11]:
lf_v2 = pl.scan_parquet(MAT_V2)

# 1. Duplicados bajo la clave de producción (K7) sobre el V2
dup_k7_v2 = (
    lf_v2
    .group_by(CLAVE_MAT_PROD)
    .len()
    .filter(pl.col("len") > 1)
    .select(pl.col("len").sum())
    .collect()
    .item()
)

print(f"Duplicados bajo clave de producción K7:")
print(f"  · original (antes de imputar): {dup_k7_orig:,}")
print(f"  · V2 (después de imputar grupos): {dup_k7_v2:,}")
if dup_k7_v2 == 0:
    print("  → 0 duplicados ✅")
else:
    print("  → >0: colisiones producidas por la imputación de grupos (se reporta, no se elimina).")

Duplicados bajo clave de producción K7:
  · original (antes de imputar): 0
  · V2 (después de imputar grupos): 13,284
  → >0: colisiones producidas por la imputación de grupos (se reporta, no se elimina).


In [12]:
# 2. Nulos en columnas imputadas (grupos, textos, año)
nulos_v2 = (
    lf_v2
    .select([
        pl.col(c).null_count().alias(c) for c in GRUPO_COLS
    ] + [
        pl.col("DEPARTAMENTO_NACIMIENTO").null_count().alias("DEPARTAMENTO_NACIMIENTO"),
        pl.col("NACIONALIDAD").null_count().alias("NACIONALIDAD"),
        pl.col("ANIO_PERIODO_INGRESO").null_count().alias("ANIO_PERIODO_INGRESO"),
    ])
    .collect()
    .transpose(include_header=True, header_name="columna", column_names=["nulos"])
)
print("Nulos en columnas imputadas (V2):")
print(nulos_v2)
assert nulos_v2.select(pl.col("nulos").sum()).item() == 0, "Quedan nulos en columnas imputadas"
print("✅ 0 nulos en todas las columnas imputadas.")

Nulos en columnas imputadas (V2):
shape: (7, 2)
┌─────────────────────────┬───────┐
│ columna                 ┆ nulos │
│ ---                     ┆ ---   │
│ str                     ┆ u32   │
╞═════════════════════════╪═══════╡
│ CODIGO_GRUPO_1          ┆ 0     │
│ NOMBRE_GRUPO_1          ┆ 0     │
│ CODIGO_GRUPO_3          ┆ 0     │
│ NOMBRE_GRUPO_3          ┆ 0     │
│ DEPARTAMENTO_NACIMIENTO ┆ 0     │
│ NACIONALIDAD            ┆ 0     │
│ ANIO_PERIODO_INGRESO    ┆ 0     │
└─────────────────────────┴───────┘
✅ 0 nulos en todas las columnas imputadas.


In [14]:
# 3. Espacios en bordes y espacios dobles en columnas Utf8 (V2)
# Se hace una única consulta para todas las columnas (una pasada, no 26)
exprs = []
for c in STRING_COLS:
    exprs.append((pl.col(c).str.starts_with(" ")).sum().alias(f"borde_ini_{c}"))
    exprs.append((pl.col(c).str.ends_with(" ")).sum().alias(f"borde_fin_{c}"))
    exprs.append(pl.col(c).str.contains("  ", literal=True).sum().alias(f"doble_{c}"))

resultado = lf_v2.select(exprs).collect()

total_borde = sum(resultado[f"borde_ini_{c}"][0] + resultado[f"borde_fin_{c}"][0] for c in STRING_COLS)
total_doble = sum(resultado[f"doble_{c}"][0] for c in STRING_COLS)

print(f"Espacios en bordes restantes (V2): {total_borde:,}")
print(f"Espacios dobles restantes (V2): {total_doble:,}")
assert total_borde == 0 and total_doble == 0, "Quedan espacios en bordes o dobles"
print("✅ 0 espacios en bordes y 0 espacios dobles en todas las columnas Utf8.")

Espacios en bordes restantes (V2): 0
Espacios dobles restantes (V2): 0
✅ 0 espacios en bordes y 0 espacios dobles en todas las columnas Utf8.


## Resumen de transformaciones

Consolidado de lo aplicado a Matriculados y estado final del archivo V2.

In [15]:
n_v2 = lf_v2.select(pl.len()).collect().item()
ancho_v2 = len(lf_v2.collect_schema().names())
tiene_disc = lf_v2.select(pl.col("TIENE_DISCAPACIDAD").sum()).collect().item()

print("=" * 72)
print("RESUMEN DE TRANSFORMACIONES — MATRICULADOS")
print("=" * 72)
print(f"Filas iniciales:                  {n_orig:,}")
print(f"Filas finales (V2):              {n_v2:,}")
print(f"Columnas finales (V2):           {ancho_v2}")
print(f"Duplicados K7 (original):        {dup_k7_orig:,}")
print(f"Duplicados K7 (V2, post-imput.): {dup_k7_v2:,}  (reportado, no eliminado)")
print(f"Grupos imputados:                {len(GRUPO_COLS)} columnas (mapa + centinela -1 / 'SIN CLASIFICACION')")
print(f"Textos imputados:                DEPARTAMENTO_NACIMIENTO y NACIONALIDAD → 'NO ESPECIFICADO'")
print(f"Año imputado:                    ANIO_PERIODO_INGRESO ← año de PERIODO_ESTANDARIZADO")
print(f"Normalización de textos:         {len(STRING_COLS)} columnas Utf8 (strip; regex solo en {len(COLS_CON_DOBLES)}: {COLS_CON_DOBLES})")
print(f"Discapacidad compactada:         {len(DISC_COLS)} columnas → TIENE_DISCAPACIDAD ({tiene_disc:,} True)")
print(f"Nulos en columnas imputadas:     0 (grupos, textos, año)")
print("=" * 72)
print(f"Archivo: {MAT_V2.name} — {MAT_V2.stat().st_size / 1e9:.2f} GB")
print(f"RSS actual: {rss_actual_gb():.2f} GB")

RESUMEN DE TRANSFORMACIONES — MATRICULADOS
Filas iniciales:                  17,368,424
Filas finales (V2):              17,368,424
Columnas finales (V2):           34
Duplicados K7 (original):        0
Duplicados K7 (V2, post-imput.): 13,284  (reportado, no eliminado)
Grupos imputados:                4 columnas (mapa + centinela -1 / 'SIN CLASIFICACION')
Textos imputados:                DEPARTAMENTO_NACIMIENTO y NACIONALIDAD → 'NO ESPECIFICADO'
Año imputado:                    ANIO_PERIODO_INGRESO ← año de PERIODO_ESTANDARIZADO
Normalización de textos:         26 columnas Utf8 (strip; regex solo en 1: ['NOMBRE_PROGRAMA'])
Discapacidad compactada:         7 columnas → TIENE_DISCAPACIDAD (60,737 True)
Nulos en columnas imputadas:     0 (grupos, textos, año)
Archivo: matriculados_clean_v2.parquet — 0.44 GB
RSS actual: 0.17 GB


In [16]:
# Limpieza: eliminar archivo intermedio y liberar memoria
MAT_STAGE1.unlink(missing_ok=True)
del lf_orig, lf_stage, lf_b, lf_v2
gc.collect()
print(f"Archivo intermedio eliminado: {'sí' if not MAT_STAGE1.exists() else 'NO'}")
print(f"Memoria RSS tras liberar Matriculados: {rss_actual_gb():.2f} GB")
print("✅ Notebook completado. Solo se procesó Matriculados.")

Archivo intermedio eliminado: sí
Memoria RSS tras liberar Matriculados: 0.17 GB
✅ Notebook completado. Solo se procesó Matriculados.
